# ScaleLab on Colab: run one experiment

This notebook only installs ScaleLab and calls `scalelab run`, the same command you run on your laptop. Everything a run writes (checkpoints, artifacts and a copy of the registry) goes to Google Drive, so it outlives the VM.

1. **Runtime → Change runtime type → T4 GPU.**
2. Fill in the settings below, then **Runtime → Run all**.
3. If the session ends before the run does, open this notebook again and **Run all** with the same settings: the run resumes from its last checkpoint (saved every 15 minutes).

In [ ]:
import os

# The code to run: a full commit SHA (`git log -1 --format=%H`) pins it exactly.
COMMIT = "main"  # @param {type:"string"}
EXPERIMENT = "configs/experiments/hello.yaml"  # @param {type:"string"}
OVERRIDES = ""  # @param {type:"string"}

os.environ["COMMIT"] = COMMIT
os.environ["EXPERIMENT"] = EXPERIMENT
os.environ["OVERRIDES"] = OVERRIDES
os.environ["SCALELAB_HOME"] = "/content/drive/MyDrive/scalelab"  # on Drive: outlives the VM

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
# Install ScaleLab at COMMIT, with exactly the locked dependencies (a few minutes: PyTorch).
set -euo pipefail
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
pip install --quiet uv
rm -rf /content/scalelab
git clone --quiet https://github.com/Argo-Navis-Lab/scalelab.git /content/scalelab
cd /content/scalelab
git checkout --quiet --detach "$COMMIT"
git log -1 --format="code: %H %s"
uv sync --locked --quiet

In [ ]:
%%bash
# Run it. The live registry stays on the VM's own disk (SQLite over the Drive mount is
# unreliable), and scalelab copies it to Drive after every change.
cd /content/scalelab
uv run scalelab run "$EXPERIMENT" $OVERRIDES runtime.local_registry=/content/registry.db

In [ ]:
%%bash
# Every run in the registry, with its grades.
cd /content/scalelab
uv run python - <<'EOF'
from scalelab.core.registry import Registry

registry = Registry.sqlite("/content/registry.db")
for run in registry.list_runs():
    print(run.run_id, run.status, run.hardware)
    for e in registry.list_evals(run.run_id):
        print(f"  step {e.step:>6}  {e.task}  {e.value:.1%} ± {e.stderr:.1%}  (n={e.n})")
EOF